___
# Week 5, notebook 1: Data split and preparation

> **Directions:**
> 
> * Step 1. Click **"Run all"** at the top to execute the whole notebook.
>
> * Step 2. Go down the page following directions below. 
>
> * Do not re-run the cells. If unsure, click on the little downward pointing arrow next to "Run all" and select "Restart session".
>
> Enjoy!

## Key learning:

After working through this notebook you will be familiar with:

* How to split the dataset
* How to impute missing values
* How to encode categorical variables
* How standardise numberic variables

## Import libraries

Here, we import open-source libraries that we will use for data transformation.
* `pandas` is a powerful and flexible framework for working with datasets
* `ipywidgets` enables interactive visualisations
* `utils` is out custom script that provides some additional functions

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
from ipywidgets import (
    Dropdown,
    Button,
    Layout,
    Output,
    Label, 
    FloatSlider, 
    HBox,
    interact,
)

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import plotly.graph_objects as go

# Pretty plots
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('ticks')
plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.title_fontsize'] = 12
plt.rcParams['legend.fontsize'] = 12

from utils import get_cols_for_display, calculate_missing_values

# Set display options and button color
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
button_color = 'lightblue'

## Load data

Data for type 2 diabetes patients has been pre-selected and saved in a separate file. 
* `parquet` files allow storing large amounts of data and associated metadata and quickly loading them in Jupyter notebooks. 
* The dataset contains patient demographics​, vitals​, edical history, including history and family history of diabetes​, problem list​, current medications​, lab results​, number of previous admissions.
* All data is synthetic and does not contain any real patient data.

In [2]:
# Load the prepared dataset with only T2DM patients
dataset = pd.read_parquet("../data/t2dm_synthetic_T2DM_only_prepared.parquet", engine="pyarrow")

## Reminder: column types

Here, we can look at variable types and understand what information is stored in each column.
* Variables are grouped by type: discrete, continuous, boolean, or categorical. Use the drop down menu to pick a variable type.
* Browse through column names and their definitions.
* You may have to rely on your clinician team members to fully understand what each column means.

In [3]:
# Map column names to types and descriptions for display
cols_for_display = get_cols_for_display(dataset)

# Interactive widget to select a type of feature and display the corresponding columns
data_types_list = Dropdown(
    options=['Discrete', 'Continuous', 'Boolean', 'Categorical', 'All'],
    description='Select a variable type:',
    disabled=False,
    style={'description_width': 'max-content'},
    value='Discrete'
)

def retrieve_cols_for_display(col_type):
    if col_type != 'All':
        return cols_for_display[(cols_for_display['type'] == col_type)]
    else:
        return cols_for_display

interact(retrieve_cols_for_display, col_type=data_types_list);

interactive(children=(Dropdown(description='Select a variable type:', options=('Discrete', 'Continuous', 'Bool…

## Split the data

Anything we learn from data on our patients needs to be carefully validated on unseen records. 
* Let's reserve part of our dataset to make sure we can later validate our learnings and assumptions. 
* This reserved cut of data is often referred to as *hold-out data*, *unseen data*, or a *blind test set* to emphasise that the records are **not seen at any point** during the exploration and development phases.
* Here, we will split our data into a *development* and *hold-out* sets. 
* Use the slider to decide how much data to allocate into each set (common split ratio is 80:20).
* We will make sure to preserve the proportion of re-admitted patients in every subset.

**Important to note:**
* Later, we will further split the *development* using cross-validation to protect ourselves from leaking any information about the data.

In [4]:
label_widget = Label('Split ratio:')
split_ratio = FloatSlider(value=0.8, min=0, max=1, step=0.05)

button = Button(description ='Randomly split data on development and hold-out sets', layout=Layout(width='50%'))
button.style.button_color = button_color
output = Output()

def on_button_clicked(_):
    output.clear_output()
    with output:
        dev_dataset, holdout_dataset = train_test_split(dataset, train_size=split_ratio.value, random_state=42, stratify=dataset.outcome_admission_in_2025)
        print("The dataset has been split into training and testing sets based on the selected split ratio:")
        print("- %d records allocated into the development set" % dev_dataset.shape[0])
        print("- %d records allocated into the hold-out set" % holdout_dataset.shape[0])

        source = [0, 0 , 1, 1 ,2 ,2]
        target = [1, 2, 3, 4, 3, 4]
        value = [
            dataset.outcome_admission_in_2025.value_counts().iloc[1], 
            dataset.outcome_admission_in_2025.value_counts().iloc[0], 
            dev_dataset.outcome_admission_in_2025.value_counts().iloc[1],
            holdout_dataset.outcome_admission_in_2025.value_counts().iloc[1],
            dev_dataset.outcome_admission_in_2025.value_counts().iloc[0],
            holdout_dataset.outcome_admission_in_2025.value_counts().iloc[0]
            ]
        color_node = "slategrey"
        color_link = sns.color_palette("tab20", 6).as_hex()
        label_sankey = ["Full dataset", "Admitted", "Not admitted", "Development set", "Hold-out set"]

        link = dict(source = source, target = target, value = value, color=color_link)
        node = dict(label = label_sankey, pad=50, thickness=5, color=color_node)
        data = go.Sankey(link = link, node=node, valueformat = ".0f",)
        fig = go.Figure(data)
        fig.update_layout(
            autosize=True,
            width=900,
            height=400,)
        fig.show()


button.on_click(on_button_clicked)

display(HBox([label_widget, split_ratio]), button)
display(output)

Button(description='Randomly split data on development and hold-out sets', layout=Layout(width='50%'), style=B…

Output()

## Missing values

Now let's have a look at how much data in our dataset is missing:
* There may be different reasons for why data is missing.
* Sometimes a value can be calcualted from other variables in the dataset.
* Other times a variable may have to be excluded from further analysis altogether.

In [5]:
# Interactive widget to show the number of missing values in each column
button2 = Button(description ='Show number of missing values in the development set', layout=Layout(width='50%'))
button2.style.button_color = button_color
output2 = Output()

def on_button_clicked2(_):
    output2.clear_output()
    # Split data
    dev_dataset, _ = train_test_split(dataset, train_size=split_ratio.value, random_state=42, stratify=dataset.outcome_admission_in_2025)
    missing_values = calculate_missing_values(dev_dataset)
    with output2:
        display(missing_values)

button2.on_click(on_button_clicked2)

display(button2)
display(output2)

Button(description='Show number of missing values in the development set', layout=Layout(width='50%'), style=B…

Output()

## Data preparation
Machine learning models have a few requirements when it comes to data:
* Some models cannot handle missing value (some models are able to ignore them)
* All inputs and outputs need to be numeric (this is maths and maths only deals with numbers!)
* Some models don't do well with variables of dramatically different scale (we need to compare apples with apples).

**Important to note:**

Some of the steps above require learning from our data, e.g. what is the median weight of our patients to use for imputation? It is cruicial to learn those values from the *development* so we are not leaking any information from the held-out, unseen records.

Below we will see how by going through these steps we can transform our dataset and prepare it for machine learning models. 

In [6]:
# Split data
dev_dataset, _ = train_test_split(dataset, train_size=split_ratio.value, random_state=42, stratify=dataset.outcome_admission_in_2025)

# Define feature names
cols_to_exclude = ['patient_id', 'outcome_admission_in_2025']
categorical_features = dev_dataset.drop(cols_to_exclude, axis=1).select_dtypes(include='category').columns.tolist()
numeric_features = dev_dataset.drop(cols_to_exclude, axis=1).select_dtypes(exclude='category').columns.tolist()

dev_dataset.head(10)

,patient_id,hospital,age,sex,height,weight,weight_over_100,bmi,waist,obesity,interpreter,alcohol,smoker,systolic_bp,diastolic_bp,elevated_bp,years_since_diagnosis,gestational_diabetes,family_history,siblings,children,hypertension,hyperlipidaemia,ischemic_heart_disease,cardiac_failure,neuropathy,nephropathy,sympt_peripheral_neuropathy,depression,stroke,acute_myocardial_infarction,transient_ischemic_attack,cabg,cardiomyopathy,autonomic_neuropathy,retinopathy,lower_limb_problems,charcot_foot,ulceration,claudication,nephropathy_indication,cardiovascular_disease,cerebrovascular_disease,oral_contraceptive,beta_blockers,ace_inhibitor,calcium_channel_blocker,corticosteroids,thaizide,agr2_receptor_blocker,aspirin,method_manage_t2dm,sodium,potassium,chloride,bicarbonate,creatinine,urea,haemoglobin,albumin,white_cell_count,platelets,red_cell_count,packed_cell_volume,mean_cell_volume,hba1c,egfr,islet_antibody,coeliac_antibody,thyroid_function,thyroid_antibody,vitamin_b12,dka_diagnosis,admissions_in_2024,visits_before_2025,outcome_admission_in_2025
748,816377,RMH,59,M,158.0,83.9,False,33.603221,79.8,True,False,False,CURRENT,125.0,80.1,False,28,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,DIET ONLY,139.0,4.6,106.9,24.4,73.2,12.4,109.6,35.0,10.5,243.0,4.2,0.3,82.4,8.9,NaN,False,False,False,False,False,False,0,7,False
1435,421757,RMH,75,M,160.0,97.3,False,38.008909,81.0,True,False,False,NO,136.7,76.0,False,4,False,False,True,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,TABLETS,140.0,4.5,104.2,24.1,196.1,25.2,112.0,NaN,10.0,238.0,2.5,0.2,95.1,8.2,NaN,False,False,False,False,False,False,0,36,True
1648,960767,RMH,71,M,173.0,97.9,False,32.706432,84.0,True,False,False,UNKNOWN,110.0,70.1,False,12,False,False,False,False,False,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,True,BOTH INSULIN AND TABS,137.0,3.8,103.9,23.4,80.2,9.4,138.0,36.0,9.0,143.0,5.7,0.4,94.7,6.9,NaN,False,False,False,False,False,False,0,38,False
74,915817,RMH,35,F,172.0,101.4,True,34.270917,92.3,True,False,False,NO,130.0,75.1,False,12,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,UNKNOWN,136.0,4.3,103.9,24.0,74.2,15.9,127.6,38.0,6.8,439.0,4.5,0.4,87.9,9.8,NaN,False,False,False,False,False,False,0,18,False
395,904071,RMH,55,M,157.0,104.3,True,42.318784,91.9,True,False,False,NO,197.8,69.2,True,12,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,NON-INSULIN,135.0,4.4,103.2,20.8,108.0,17.4,93.9,21.3,9.5,478.0,2.5,0.2,97.7,11.5,NaN,False,False,False,False,False,False,2,21,True
258,504593,WH,47,M,165.0,93.6,False,34.375419,83.8,True,False,False,NO,NaN,NaN,False,12,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,INSULIN,140.0,4.4,101.9,25.0,89.2,5.6,163.8,41.0,8.3,299.0,5.1,0.4,86.0,8.3,NaN,False,False,False,False,False,False,0,2,False
230,649558,RMH,46,M,166.0,65.9,False,23.910249,70.3,False,False,False,UNKNOWN,NaN,NaN,False,14,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,UNKNOWN,139.0,3.7,103.9,23.7,64.2,2.0,157.2,35.0,5.9,243.0,4.3,0.4,83.2,7.5,NaN,False,False,False,False,F

### Impute missing values

When handling missing values, we need to decide with which values to fill in. This can be the mean or the median for numeric variables, the most frequent value for categorical variables; we could define a constant or try and model missing values using other columns of the dataset (e.g., calculate BMI from weight and height).
* Here, we will impute all numeric variables with their respective medians.
* For the categoricals, we will calculate the most frequent value per column.

We can "remember" the median and most frequent values we have calculated from our *development* set and use them later to also impute missing values in the *hold-out* set.

> `scikit-learn` is a powerful Python library most commonly used to develop machine learning models. 
>
> It has a large number of modules to support different stages of data preparation and model development. The library was created and is maintained by a community of software engineers, developers, and researchers and is an incredible example of how community-driven projects can grow into becoming the core tool for the majority of machine learning and AI coders and scientists today.

In [ ]:
# Interactive widget to impute missing values
button3 = Button(description ='Impute missing values in the development set', layout=Layout(width='50%'))
button3.style.button_color = button_color
output3 = Output()

def on_button_clicked3(_):
    output3.clear_output()
    # Define transformers for numeric and categorical features. Adding SimpleImputer to handle the missing values in the dataset.
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy='most_frequent')), 
        ]
    )

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy='median')), 
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", categorical_transformer, categorical_features),
            ("numeric", numeric_transformer, numeric_features)
        ]
    )
    # Fit the preprocessor
    preprocessor.fit(dev_dataset)
    # Transform the development set
    dev_dataset_transformed = pd.concat([
        dev_dataset[cols_to_exclude],
        pd.DataFrame(preprocessor.transform(dev_dataset), index=dev_dataset.index, columns=categorical_features + numeric_features),
        ], axis=1)
    missing_values = calculate_missing_values(dev_dataset_transformed.drop(cols_to_exclude, axis=1), report_zeros=True)
    with output3:
        display(dev_dataset_transformed.loc[:, cols_to_exclude + ['hospital', 'age', 'sex'] + numeric_features[1:]].head(10))
        print("\nProportion of missing values after imputation:")
        display(missing_values)


button3.on_click(on_button_clicked3)

display(button3)
display(output3)

Button(description='Impute missing values in the development set', layout=Layout(width='50%'), style=ButtonSty…

Output()

### Encode categorical variables

We have looked at couple of approaches to encoding last week.
* Here, we will use already familiar one-hot encoding to replace our categorical variables with a set of boolen columns.
* We will again "remember" which categories we have encountered in the *development* set and will expect the same values in the *holdout* set.
* Any new categories in unseen records will be ignored setting all columns to 0. 

In [15]:
# Interactive widget to encode categorical variables
button4 = Button(description ='Encode categorical variables in the development set', layout=Layout(width='50%'))
button4.style.button_color = button_color
output4 = Output()

def on_button_clicked4(_):
    output4.clear_output()
    # Define transformers for numeric and categorical features. Adding OneHotEncoding to convert categorical variables into numbers.
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy='most_frequent')), 
            ("encoder", OneHotEncoder(drop=None, sparse_output=False, handle_unknown='ignore')),
        ]
    )

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy='median')), 
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", categorical_transformer, categorical_features),
            ("numeric", numeric_transformer, numeric_features)
        ]
    )
    # Fit the preprocessor
    preprocessor.fit(dev_dataset)
    categorical_features_encoded = list(np.hstack(preprocessor.transformers_[0][1][1].categories_))
    # Transform the development set
    dev_dataset_transformed = pd.concat([
        dev_dataset[cols_to_exclude],
        pd.DataFrame(preprocessor.transform(dev_dataset), index=dev_dataset.index, columns=categorical_features_encoded + numeric_features),
        ], axis=1)
    with output4:
        display(dev_dataset_transformed.loc[:, cols_to_exclude + ['RMH', 'WH', 'age', 'F', 'M'] + numeric_features[1:]].head(10))
        
button4.on_click(on_button_clicked4)

display(button4)
display(output4)

Button(description='Encode categorical variables in the development set', layout=Layout(width='50%'), style=Bu…

Output()

### Standardise numeric variables

*[Remember, our categorical variables are now also numeric]*

Some machine learning models compare records based on differences rather than absolute values. What does this mean? 
* Imagine you have two patients, one got readmitted and the other one did not.
* The other patient is 173 cm tall and has hba1c of 11.5%
* One patient is 157 cm tall and has hba1c of 5.9%.
* The difference in height is 16cm and the difference in hba1c 5.6.
* The difference in height is not likely to have anything to do with the risk of readmission, while the higher hba1c definitely has. The model however does not have this domain knowledge and might assume features with larger differences are more reliable in distinguishing the two groups of patients.

Here, we will bring all numeric variables to the same scale by subtracting from each column its mean and dividing it by its standard deviation. We will, as usual, "remember" those means and standard deviations and use these statistics to standardise the variables in the *holdout* set.

In [ ]:
# Interactive widget to standardise numeric variables
button5 = Button(description ='Standardise numeric variables in the development set', layout=Layout(width='50%'))
button5.style.button_color = button_color
output5 = Output()

def on_button_clicked5(_):
    output5.clear_output()
    # Define transformers for numeric and categorical features. Adding StandardScaler to standardise numeric variables.
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy='most_frequent')), 
            ("encoder", OneHotEncoder(drop=None, sparse_output=False, handle_unknown='ignore')),
            ("scaler", StandardScaler()),
        ]
    )

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy='median')), 
            ("scaler", StandardScaler()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", categorical_transformer, categorical_features),
            ("numeric", numeric_transformer, numeric_features)
        ]
    )
    # Fit the preprocessor
    preprocessor.fit(dev_dataset)
    categorical_features_encoded = list(np.hstack(preprocessor.transformers_[0][1][1].categories_))
    # Transform the development set
    dev_dataset_transformed = pd.concat([
        dev_dataset[cols_to_exclude],
        pd.DataFrame(preprocessor.transform(dev_dataset), index=dev_dataset.index, columns=categorical_features_encoded + numeric_features),
        ], axis=1)
    with output5:
        display(dev_dataset_transformed.loc[:, cols_to_exclude + ['RMH', 'WH', 'age', 'F', 'M'] + numeric_features[1:]].head(10))
        
button5.on_click(on_button_clicked5)

display(button5)
display(output5)

Button(description='Standardise numeric variables in the development set', layout=Layout(width='50%'), style=B…

Output()